In [1]:
import os
import pandas as pd
import numpy as np
import glob

import utilities as utils

In [2]:
save_data_flag = True

## Bead prediction task

In [3]:
beads_datadir = "./Data/beads/"
beads_rawdatadir = beads_datadir + "raw/"
sjlist = pd.read_csv(beads_datadir + 'sj_list.csv')
# list of subject indices for which the choice data was flipped relative to the jar and bead data
with open(beads_datadir + 'flipped_sjs.txt', 'r',  encoding='utf-16') as f:
    flipped_sjs = [int(line.strip()) for line in f.readlines()]

wsize = 7 # number of observed beads to include in the "window" used to compute choice probabilities and posterior probabilities for each unique X (sequence of observed beads of length wsize, from current trial backwards)
ref_choice = 0 # choice (0 or 1) to use as reference for computing choice probabilities for each unique X (e.g. P(choice=ref_choice | X)) and for computing posterior probabilities of the hidden state (jar) given X (e.g. P(jar=ref_choice | X))

# generative parameters for the bead-prediction experiment
h_=0.99 # jar stay probability
p0_=0.8 # probability of drawing bead type 0 from jar 0
p1_=0.2 # probability of drawing bead type 0 from jar 1
P0 = np.array([[0.5,0.5]]) # prior over jars for computing posterior probabilities over hidden markov process
H = np.ones((2,2)) - np.abs(np.eye(2)*-1 + h_) # transition matrix
E = np.vstack((np.array([[1,0]])*p0_ + np.array([[0,1]])*(1-p0_),np.array([[1,0]])*p1_ + np.array([[0,1]])*(1-p1_))) # emission matrix

datafiles = [f for f in os.listdir(beads_rawdatadir) if os.path.isfile(beads_rawdatadir + f)]

# make a csv with trial-wise data for each subject, to be used for pre-processing and analyses
preprocdf = pd.DataFrame({
    'subject_ID': [],
    'subject_index': [],
    'run': [],
    'trial': [],
    'jar': [],
    'bead': [],
    'choice': []
})

for ii,sub in enumerate(datafiles):

    df = pd.read_csv(beads_rawdatadir + sub)

    # necessary to ensure consistent ordering of sj index
    sjid = df['subject_id'].iloc[0]
    sjind = sjlist[sjlist['subject_ID']==sjid]['subject_index'].iloc[0]

    datadf = df[df['block'] == 'lowH']

    for run in [1,2]:
        choices = datadf.loc[datadf['block_rep'] == run, 'choice'].to_numpy()
        jars = datadf.loc[datadf['block_rep'] == run, 'jar'].to_numpy()
        beads = datadf.loc[datadf['block_rep'] == run, 'bead'].to_numpy()
        trials = datadf.loc[datadf['block_rep'] == run, 'trial_number'].to_numpy()

        sjdf = pd.DataFrame({
            'subject_ID': [sjid] * len(trials),
            'subject_index': [sjind] * len(trials),
            'run': [run] * len(trials),
            'trial': trials,
            'jar': jars,
            'bead': beads,
            'choice': choices
        })

        preprocdf = pd.concat((preprocdf,sjdf),ignore_index=True)

preprocdf.sort_values(by=['subject_index','run','trial'],inplace=True,ignore_index=True)

# sequence of trials (bead draws and hidden jars) that every subject experienced (both run 1 and run 2 had the same sequence of trials)
trialseq_df = preprocdf.loc[(preprocdf['run']==1) & (preprocdf['subject_index']==0)][['trial','jar','bead']]

# make a csv that gives the choice probabilities for each unique X (sequence of observed beads of length window_size, from current trial backwards)
choiceprobdf = pd.DataFrame({
    'subject_ID': [],
    'subject_index': [],
    'run': [],
    'window_size': [],  # window size used to compute posterior probabilities and determine unique Xs
    'X': [],
    'Ntrials': [],  # number of trials of a given unique X
    'posterior_prob': [],
    'choice_prob': []
})

Xemp, Yemp = utils.getXY_beads(trialseq_df['bead'].to_numpy(), trialseq_df['jar'].to_numpy(), wsize)
Xemp_enc, _ = utils.getXY_beads(trialseq_df['bead'].to_numpy(), trialseq_df['jar'].to_numpy(), wsize, encodeX=True)
p_YgX = utils.P_jar_g_beads(Xemp,E,H,P0)
Xunique = np.unique(Xemp_enc)

for sjind in preprocdf['subject_index'].unique():

    sjid = preprocdf[preprocdf['subject_index']==sjind]['subject_ID'].iloc[0]

    for run in [1,2]:

        sjconds = (preprocdf['run']==run) & (preprocdf['subject_index']==sjind)
        choices = utils.getR_beads(preprocdf[sjconds]['choice'].to_numpy(),wsize)
        if sjind in flipped_sjs:
            choices = 1 - choices

        for Xun in Xunique:

            Xinds = np.where(Xemp_enc == Xun)[0]

            choice_prob = np.mean(choices[Xinds]==ref_choice)

            sjdf = pd.DataFrame({
                'subject_ID': [sjid],
                'subject_index': [sjind],
                'run': [run],
                'window_size': [wsize],
                'X': [Xun],
                'Ntrials': [len(Xinds)],
                'posterior_prob': [p_YgX[Xinds[0],ref_choice]],
                'choice_prob': [choice_prob]
            })

            choiceprobdf = pd.concat((choiceprobdf,sjdf),ignore_index=True)

# make csv giving the posterior probability of the hidden state (jar) given each unique X for a given strategy - e.g., "fully-optimal" (7-back) strategy, or "1-back" strategy, etc.
postprobdf = pd.DataFrame({
    'strategy': [],
    'X': [],
    'posterior_prob': []
})

Xemp1b, Yemp1b = utils.getXY_beads(trialseq_df['bead'].to_numpy(), trialseq_df['jar'].to_numpy(), wsize=1,ref_wsize=wsize)
p_YgX1b = utils.P_jar_g_beads(Xemp1b,E,H,P0)
for Xun in Xunique:
    Xinds = np.where(Xemp_enc == Xun)[0]
    sjdf = pd.DataFrame({
        'strategy': ['fully-optimal'],
        'X': [Xun],
        'posterior_prob': [p_YgX[Xinds[0],ref_choice]]
    })
    postprobdf = pd.concat((postprobdf,sjdf),ignore_index=True)
    sjdf1b = pd.DataFrame({
        'strategy': ['1-back'],
        'X': [Xun],
        'posterior_prob': [p_YgX1b[Xinds[0],ref_choice]]
    })
    postprobdf = pd.concat((postprobdf,sjdf1b),ignore_index=True)

if save_data_flag:
    preprocdf.to_csv(beads_datadir + 'sj-preproc-data.csv',index=False)
    trialseq_df.to_csv(beads_datadir + 'trial_sequence.csv',index=False)
    choiceprobdf.to_csv(beads_datadir + 'sj-psychometric-data.csv',index=False)
    postprobdf.to_csv(beads_datadir + 'strat-post-probs.csv',index=False)

## Horse prediction experiments

In [4]:
horses_datadir = "./Data/horses/"
ref_choice = 0 # choice (0 or 1) to use as reference for computing choice probabilities for each unique X (e.g. P(choice=ref_choice | X)) and for computing posterior probabilities of the hidden state (jar) given X (e.g. P(jar=ref_choice | X))

paramdict = {
    'lowWS': {
        'weakLLR': 0.45,
        'WSratio': 1.3,
        'p1': 0.06
    },
    'midWS': {
        'weakLLR': 0.2,
        'WSratio': 2.5,
        'p1': 0.08
    },
    'highWS': {
        'weakLLR': 0.18,
        'WSratio': 6.3,
        'p1': 0.02
    }
}

p_Y = np.array([[0.5, 0.5]])

### Low WS ratio experiment

In [5]:
horses_datadir_low = horses_datadir + "lowWS/"
sjlist = pd.read_csv(horses_datadir_low + 'sj_list.csv')

fulldf = pd.DataFrame({
    'subject_ID': [],
    'subject_index': [],
    'exclude_Ixr': [],
    'trial_ID': [],
    'trial_index': [],
    'observation_1': [],
    'observation_2': [],
    'observation_3': [],
    'observation_4': [],
    'observation_5': [],
    'latent_state': [],
    'optimal_choice': [],
    'correct': [],
    'intertrial_duration': [],
    'trial_start_time': [],
    'choice': [],
    'rt': []
})

run1_columns = [
    'trial_index',
    'shape1',
    'shape2',
    'shape3',
    'shape4',
    'shape5',
    'distr',
    'pred_distr',
    'num_completed',
    'key_resp_test.corr',
    'key_resp_6.rt',
    'testing_trial.started',
    'choice', # added in the loop below
    'key_resp_test.rt'
]

renamed_columns = [
    'trial_ID',
    'observation_1',
    'observation_2',
    'observation_3',
    'observation_4',
    'observation_5',
    'latent_state',
    'optimal_choice',
    'trial_index',
    'correct',
    'intertrial_duration',
    'trial_start_time',
    'choice',
    'rt'
]

run1_map = dict(zip(run1_columns,renamed_columns))

shape_map = {
    'shape_1': 1.0,
    'shape_2': 2.0,
    'shape_3': 3.0,
    'shape_4': 4.0
}

shape_map_all = {
    'observation_1': shape_map,
    'observation_2': shape_map,
    'observation_3': shape_map,
    'observation_4': shape_map,
    'observation_5': shape_map
}

dir_list = [
    horses_datadir_low + 'raw/*.csv',
    horses_datadir_low + 'raw/exclude/*.csv',
]

sjindex = 0
dirindex = 0

for dir_ in dir_list:
    for fname in glob.glob(dir_):

        sjdf = pd.DataFrame({})

        df = pd.read_csv(fname)
        sjid = df['PROLIFIC_PID'].iloc[0]
        dist2_side = df['dist2_loc'].iloc[0]
        # if df['dist1_loc'].iloc[0] == 'left':
        #     choice_map = dict({'left': 1, 'right': 2})
        # else:
        #     choice_map = dict({'right': 1, 'left': 2})

        # run 1
        datadf = df[df['key_resp_test.keys'].notna()]
        datadf['choice'] = datadf.loc[:,'key_resp_test.keys'] == dist2_side
        datadf = datadf[run1_columns]
        datadf.rename(columns=run1_map,inplace=True)
        sjdf = pd.concat((sjdf,datadf),ignore_index=True)

        sjdf['choice'] = sjdf['choice'].astype(int)
        sjdf['subject_ID'] = sjid
        sjdf['subject_index'] = sjlist[sjlist['subject_ID']==sjid]['subject_index'].iloc[0]
        sjdf['trial_index'] = sjdf['trial_index'] - 100
        if dirindex > 0:
            sjdf['exclude_Ixr'] = 1
        else:
            sjdf['exclude_Ixr'] = 0

        sjindex += 1

        fulldf = pd.concat((fulldf,sjdf),ignore_index=True)

    dirindex += 1

fulldf.replace(shape_map_all,inplace=True)
fulldf['latent_state'] = fulldf['latent_state'] - 1
fulldf['optimal_choice'] = fulldf['optimal_choice'] - 1

fulldf['observation_encoding'] = \
    10**(fulldf['observation_1']-1) + \
    10**(fulldf['observation_2']-1) + \
    10**(fulldf['observation_3']-1) + \
    10**(fulldf['observation_4']-1) + \
    10**(fulldf['observation_5']-1)

fulldf['observation_encoding_ew'] = \
    10**(fulldf['observation_1']>2) + \
    10**(fulldf['observation_2']>2) + \
    10**(fulldf['observation_3']>2) + \
    10**(fulldf['observation_4']>2) + \
    10**(fulldf['observation_5']>2)

no_strong_flag = \
    ~( \
    (fulldf['observation_1']==1) | (fulldf['observation_1']==4) | \
    (fulldf['observation_2']==1) | (fulldf['observation_2']==4) | \
    (fulldf['observation_3']==1) | (fulldf['observation_3']==4) | \
    (fulldf['observation_4']==1) | (fulldf['observation_4']==4) | \
    (fulldf['observation_5']==1) | (fulldf['observation_5']==4) \
        )

fulldf['observation_encoding_iw'] = \
    (10**(fulldf['observation_1']-1)) * ((fulldf['observation_1']==1) | (fulldf['observation_1']==4)+no_strong_flag) + \
    (10**(fulldf['observation_2']-1)) * ((fulldf['observation_2']==1) | (fulldf['observation_2']==4)+no_strong_flag) + \
    (10**(fulldf['observation_3']-1)) * ((fulldf['observation_3']==1) | (fulldf['observation_3']==4)+no_strong_flag) + \
    (10**(fulldf['observation_4']-1)) * ((fulldf['observation_4']==1) | (fulldf['observation_4']==4)+no_strong_flag) + \
    (10**(fulldf['observation_5']-1)) * ((fulldf['observation_5']==1) | (fulldf['observation_5']==4)+no_strong_flag)

fulldf.sort_values(by=['subject_index','trial_index'],inplace=True,ignore_index=True)

# make a csv of the trial set (horses and shape combinations) that every subject saw
trialset_df = fulldf.loc[(fulldf['subject_index']==0)][['trial_ID','observation_encoding','latent_state']].sort_values(by='trial_ID',ignore_index=True).copy()

# make a csv that gives the choice probabilities for each unique X (shape combination) for each subject
choiceprobdf = pd.DataFrame({
    'subject_ID': [],
    'subject_index': [],
    'X': [],
    'Ntrials': [],  # number of trials of a given unique X
    'posterior_prob': [],
    'choice_prob': []
})

Xemp = utils.split_to_four_digits(trialset_df['observation_encoding'].to_numpy())
Xemp_enc = trialset_df['observation_encoding'].to_numpy()
Yemp = trialset_df['latent_state'].to_numpy()
p_YgX = utils.P_horse_g_shapecomb(Xemp,paramdict['lowWS']['weakLLR'],paramdict['lowWS']['WSratio'],paramdict['lowWS']['p1'],p_Y)
Xunique = np.unique(Xemp_enc)

for sjind in fulldf['subject_index'].unique():

    sjconds = fulldf['subject_index']==sjind
    sjid = fulldf[sjconds]['subject_ID'].iloc[0]

    Xemp_sj = utils.split_to_four_digits(fulldf[sjconds]['observation_encoding'].to_numpy())
    Xemp_sj_enc = fulldf[sjconds]['observation_encoding'].to_numpy()
    Yemp_sj = fulldf[sjconds]['latent_state'].to_numpy()
    choices = fulldf[sjconds]['choice'].to_numpy()

    # for Xun in Xunique:
    for Xun in np.unique(Xemp_sj_enc):

        Xinds = np.where(Xemp_sj_enc == Xun)[0]
        Xinds2 = np.where(Xemp_enc == Xun)[0]

        choice_prob = np.mean(choices[Xinds]==ref_choice)

        sjdf = pd.DataFrame({
            'subject_ID': [sjid],
            'subject_index': [sjind],
            'X': [Xun],
            'Ntrials': [len(Xinds)],
            'posterior_prob': [p_YgX[Xinds2[0],ref_choice]],
            'choice_prob': [choice_prob]
        })

        choiceprobdf = pd.concat((choiceprobdf,sjdf),ignore_index=True)


# make csv giving the posterior probability of the hidden state (horse) given each unique X for a given strategy - e.g., optimal strategy, or equal-weights strategy, etc.
postprobdf = pd.DataFrame({
    'strategy': [],
    'X': [],
    'posterior_prob': []
})

p_YgX_ew = utils.P_horse_g_shapecomb(Xemp,paramdict['lowWS']['weakLLR'],paramdict['lowWS']['WSratio'],paramdict['lowWS']['p1'],p_Y,utils.P_shapecomb_g_horse_ew)
p_YgX_iw = utils.P_horse_g_shapecomb(Xemp,paramdict['lowWS']['weakLLR'],paramdict['lowWS']['WSratio'],paramdict['lowWS']['p1'],p_Y,utils.P_shapecomb_g_horse_iw)

for Xun in Xunique:
    Xinds = np.where(Xemp_enc == Xun)[0]
    sjdf = pd.DataFrame({
        'strategy': ['fully-optimal'],
        'X': [Xun],
        'posterior_prob': [p_YgX[Xinds[0],ref_choice]]
    })
    postprobdf = pd.concat((postprobdf,sjdf),ignore_index=True)
    sjdf_ew = pd.DataFrame({
        'strategy': ['equal-weights'],
        'X': [Xun],
        'posterior_prob': [p_YgX_ew[Xinds[0],ref_choice]]
    })
    postprobdf = pd.concat((postprobdf,sjdf_ew),ignore_index=True)
    sjdf_iw = pd.DataFrame({
        'strategy': ['ignore-weak'],
        'X': [Xun],
        'posterior_prob': [p_YgX_iw[Xinds[0],ref_choice]]
    })
    postprobdf = pd.concat((postprobdf,sjdf_iw),ignore_index=True)
    

if save_data_flag:
    fulldf.to_csv(horses_datadir_low + 'sj-preproc-data.csv',index=False)
    trialset_df.to_csv(horses_datadir_low + 'trial_set.csv',index=False)
    choiceprobdf.to_csv(horses_datadir_low + 'sj-psychometric-data.csv',index=False)
    postprobdf.to_csv(horses_datadir_low + 'strat-post-probs.csv',index=False)

C:\Users\parja\AppData\Local\Temp\ipykernel_31780\1018963567.py:98: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  datadf['choice'] = datadf.loc[:,'key_resp_test.keys'] == dist2_side
C:\Users\parja\AppData\Local\Temp\ipykernel_31780\1018963567.py:98: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  datadf['choice'] = datadf.loc[:,'key_resp_test.keys'] == dist2_side
C:\Users\parja\AppData\Local\Temp\ipykernel_31780\1018963567.py:98: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a

### Intermediate WS ratio / learning experiment

In [6]:
horses_datadir_mid = horses_datadir + "midWS_learning/"
sjlist = pd.read_csv(horses_datadir_mid + 'sj_list.csv')

fulldf = pd.DataFrame({
    'subject_ID': [],
    'subject_index': [],
    'age': [],
    'exclude_Ixr': [],
    'condition': [],
    'trial_ID': [],
    'trial_index': [],
    'observation_1': [],
    'observation_2': [],
    'observation_3': [],
    'observation_4': [],
    'observation_5': [],
    'latent_state': [],
    'optimal_choice': [],
    'correct': [],
    'intertrial_duration': [],
    'trial_start_time': [],
    'choice': [],
    'rt': []
})

run1_columns = [
    'trial_index',
    'shape1',
    'shape2',
    'shape3',
    'shape4',
    'shape5',
    'distr',
    'pred_distr',
    'num_completed_1',
    'key_resp_test.corr',
    'key_proceed_1.rt',
    'testing_trial.started',
    'choice', # added in the loop below
    'key_resp_test.rt'
]

run2_columns = [
    'trial_index',
    'shape1',
    'shape2',
    'shape3',
    'shape4',
    'shape5',
    'distr',
    'pred_distr',
    'num_completed_2',
    'key_resp_test_5.corr',
    'key_proceed_2.rt',
    'testing_trial_2.started',
    'choice',
    'key_resp_test_5.rt'
]

renamed_columns = [
    'trial_ID',
    'observation_1',
    'observation_2',
    'observation_3',
    'observation_4',
    'observation_5',
    'latent_state',
    'optimal_choice',
    'trial_index',
    'correct',
    'intertrial_duration',
    'trial_start_time',
    'choice',
    'rt'
]

run1_map = dict(zip(run1_columns,renamed_columns))
run2_map = dict(zip(run2_columns,renamed_columns))

shape_map = {
    'shape_1': 1.0,
    'shape_2': 2.0,
    'shape_3': 3.0,
    'shape_4': 4.0
}

shape_map_all = {
    'observation_1': shape_map,
    'observation_2': shape_map,
    'observation_3': shape_map,
    'observation_4': shape_map,
    'observation_5': shape_map
}

sjindex = 0

# get non-excluded subjects
for ii, fname in enumerate(glob.glob(horses_datadir_mid + 'raw/*.csv')):

    sjdf = pd.DataFrame({})

    df = pd.read_csv(fname)
    sjid = df['PROLIFIC_PID'].iloc[0]
    dist2_side = df['dist2_loc'].iloc[0]
    age = df['info_survey.question6'].iloc[0]
    # if df['dist1_loc'].iloc[0] == 'left':
    #     choice_map = dict({'left': 1, 'right': 2})
    # else:
    #     choice_map = dict({'right': 1, 'left': 2})

    # run 1
    datadf = df[df['key_resp_test.keys'].notna()]
    datadf['choice'] = datadf.loc[:,'key_resp_test.keys'] == dist2_side
    datadf = datadf[run1_columns]
    datadf.rename(columns=run1_map,inplace=True)
    datadf['condition'] = 'run_1'
    sjdf = pd.concat((sjdf,datadf),ignore_index=True)

    # run 2
    datadf = df[df['key_resp_test_5.keys'].notna()]
    datadf['choice'] = datadf.loc[:,'key_resp_test_5.keys'] == dist2_side
    datadf = datadf[run2_columns]
    datadf.rename(columns=run2_map,inplace=True)
    datadf['condition'] = 'run_2'
    sjdf = pd.concat((sjdf,datadf),ignore_index=True)
    
    sjdf['choice'] = sjdf['choice'].astype(int)
    sjdf['subject_ID'] = sjid
    sjdf['subject_index'] = sjlist[sjlist['subject_ID']==sjid]['subject_index'].iloc[0]
    sjdf['age'] = age
    sjdf['exclude_Ixr'] = 0

    sjindex += 1

    fulldf = pd.concat((fulldf,sjdf),ignore_index=True)

# get subjects excluded for information capacity < 0.05 for both conditions
for fname in glob.glob(horses_datadir_mid + 'raw/exclude/*.csv'):
    
    sjdf = pd.DataFrame({})

    df = pd.read_csv(fname)
    sjid = df['PROLIFIC_PID'].iloc[0]
    dist2_side = df['dist2_loc'].iloc[0]
    age = df['info_survey.question6'].iloc[0]
    # if df['dist1_loc'].iloc[0] == 'left':
    #     choice_map = dict({'left': 1, 'right': 2})
    # else:
    #     choice_map = dict({'right': 1, 'left': 2})

    # run 1
    datadf = df[df['key_resp_test.keys'].notna()]
    datadf['choice'] = datadf.loc[:,'key_resp_test.keys'] == dist2_side
    datadf = datadf[run1_columns]
    datadf.rename(columns=run1_map,inplace=True)
    datadf['condition'] = 'run_1'
    sjdf = pd.concat((sjdf,datadf),ignore_index=True)

    # run 2
    datadf = df[df['key_resp_test_5.keys'].notna()]
    datadf['choice'] = datadf.loc[:,'key_resp_test_5.keys'] == dist2_side
    datadf = datadf[run2_columns]
    datadf.rename(columns=run2_map,inplace=True)
    datadf['condition'] = 'run_2'
    sjdf = pd.concat((sjdf,datadf),ignore_index=True)

    sjdf['choice'] = sjdf['choice'].astype(int)
    sjdf['subject_ID'] = sjid
    sjdf['subject_index'] = sjlist[sjlist['subject_ID']==sjid]['subject_index'].iloc[0]
    sjdf['age'] = age
    sjdf['exclude_Ixr'] = 1

    sjindex += 1

    fulldf = pd.concat((fulldf,sjdf),ignore_index=True)

fulldf.replace(shape_map_all,inplace=True)
fulldf['latent_state'] = fulldf['latent_state'] - 1
fulldf['optimal_choice'] = fulldf['optimal_choice'] - 1

fulldf['observation_encoding'] = \
    10**(fulldf['observation_1']-1) + \
    10**(fulldf['observation_2']-1) + \
    10**(fulldf['observation_3']-1) + \
    10**(fulldf['observation_4']-1) + \
    10**(fulldf['observation_5']-1)

fulldf['observation_encoding_ew'] = \
    10**(fulldf['observation_1']>2) + \
    10**(fulldf['observation_2']>2) + \
    10**(fulldf['observation_3']>2) + \
    10**(fulldf['observation_4']>2) + \
    10**(fulldf['observation_5']>2)

no_strong_flag = \
    ~( \
    (fulldf['observation_1']==1) | (fulldf['observation_1']==4) | \
    (fulldf['observation_2']==1) | (fulldf['observation_2']==4) | \
    (fulldf['observation_3']==1) | (fulldf['observation_3']==4) | \
    (fulldf['observation_4']==1) | (fulldf['observation_4']==4) | \
    (fulldf['observation_5']==1) | (fulldf['observation_5']==4) \
        )

fulldf['observation_encoding_iw'] = \
    (10**(fulldf['observation_1']-1)) * ((fulldf['observation_1']==1) | (fulldf['observation_1']==4)+no_strong_flag) + \
    (10**(fulldf['observation_2']-1)) * ((fulldf['observation_2']==1) | (fulldf['observation_2']==4)+no_strong_flag) + \
    (10**(fulldf['observation_3']-1)) * ((fulldf['observation_3']==1) | (fulldf['observation_3']==4)+no_strong_flag) + \
    (10**(fulldf['observation_4']-1)) * ((fulldf['observation_4']==1) | (fulldf['observation_4']==4)+no_strong_flag) + \
    (10**(fulldf['observation_5']-1)) * ((fulldf['observation_5']==1) | (fulldf['observation_5']==4)+no_strong_flag)

fulldf.sort_values(by=['subject_index','condition','trial_index'],inplace=True,ignore_index=True)

# make a csv of the trial set (horses and shape combinations) that every subject saw
trialset_df = fulldf.loc[(fulldf['subject_index']==0) & (fulldf['condition']=='run_1')][['trial_ID','observation_encoding','latent_state']].sort_values(by='trial_ID',ignore_index=True).copy()

# make a csv that gives the choice probabilities for each unique X (shape combination) for each subject
choiceprobdf = pd.DataFrame({
    'subject_ID': [],
    'subject_index': [],
    'condition': [],
    'X': [],
    'Ntrials': [],  # number of trials of a given unique X
    'posterior_prob': [],
    'choice_prob': []
})

Xemp = utils.split_to_four_digits(trialset_df['observation_encoding'].to_numpy())
Xemp_enc = trialset_df['observation_encoding'].to_numpy()
Yemp = trialset_df['latent_state'].to_numpy()
p_YgX = utils.P_horse_g_shapecomb(Xemp,paramdict['midWS']['weakLLR'],paramdict['midWS']['WSratio'],paramdict['midWS']['p1'],p_Y)
Xunique = np.unique(Xemp_enc)

# get choice probabilities across all trials (both run 1 and run 2) for each subject, for each unique X
for sjind in fulldf['subject_index'].unique():

    sjconds = fulldf['subject_index']==sjind
    sjid = fulldf[sjconds]['subject_ID'].iloc[0]

    Xemp_sj = utils.split_to_four_digits(fulldf[sjconds]['observation_encoding'].to_numpy())
    Xemp_sj_enc = fulldf[sjconds]['observation_encoding'].to_numpy()
    choices = fulldf[sjconds]['choice'].to_numpy()

    # for Xun in Xunique:
    for Xun in np.unique(Xemp_sj_enc):

        Xinds = np.where(Xemp_sj_enc == Xun)[0]
        Xinds2 = np.where(Xemp_enc == Xun)[0]

        choice_prob = np.mean(choices[Xinds]==ref_choice)

        sjdf = pd.DataFrame({
            'subject_ID': [sjid],
            'subject_index': [sjind],
            'condition': ['all'],
            'X': [Xun],
            'Ntrials': [len(Xinds)],
            'posterior_prob': [p_YgX[Xinds2[0],ref_choice]],
            'choice_prob': [choice_prob]
        })

        choiceprobdf = pd.concat((choiceprobdf,sjdf),ignore_index=True)

# get choice probabilities separately for run 1 and run 2 for each subject, for each unique X
for sjind in fulldf['subject_index'].unique():

    for cond in ['run_1', 'run_2']:

        sjconds = (fulldf['subject_index']==sjind) & (fulldf['condition']==cond)
        sjid = fulldf[sjconds]['subject_ID'].iloc[0]

        Xemp_sj = utils.split_to_four_digits(fulldf[sjconds]['observation_encoding'].to_numpy())
        Xemp_sj_enc = fulldf[sjconds]['observation_encoding'].to_numpy()
        choices = fulldf[sjconds]['choice'].to_numpy()

        # for Xun in Xunique:
        for Xun in np.unique(Xemp_sj_enc):

            Xinds = np.where(Xemp_sj_enc == Xun)[0]
            Xinds2 = np.where(Xemp_enc == Xun)[0]

            choice_prob = np.mean(choices[Xinds]==ref_choice)

            sjdf = pd.DataFrame({
                'subject_ID': [sjid],
                'subject_index': [sjind],
                'condition': [cond],
                'X': [Xun],
                'Ntrials': [len(Xinds)],
                'posterior_prob': [p_YgX[Xinds2[0],ref_choice]],
                'choice_prob': [choice_prob]
            })

            choiceprobdf = pd.concat((choiceprobdf,sjdf),ignore_index=True)

# make csv giving the posterior probability of the hidden state (horse) given each unique X for a given strategy - e.g., optimal strategy, or equal-weights strategy, etc.
postprobdf = pd.DataFrame({
    'strategy': [],
    'X': [],
    'posterior_prob': []
})

p_YgX_ew = utils.P_horse_g_shapecomb(Xemp,paramdict['midWS']['weakLLR'],paramdict['midWS']['WSratio'],paramdict['midWS']['p1'],p_Y,utils.P_shapecomb_g_horse_ew)
p_YgX_iw = utils.P_horse_g_shapecomb(Xemp,paramdict['midWS']['weakLLR'],paramdict['midWS']['WSratio'],paramdict['midWS']['p1'],p_Y,utils.P_shapecomb_g_horse_iw)

for Xun in Xunique:
    Xinds = np.where(Xemp_enc == Xun)[0]
    sjdf = pd.DataFrame({
        'strategy': ['fully-optimal'],
        'X': [Xun],
        'posterior_prob': [p_YgX[Xinds[0],ref_choice]]
    })
    postprobdf = pd.concat((postprobdf,sjdf),ignore_index=True)
    sjdf_ew = pd.DataFrame({
        'strategy': ['equal-weights'],
        'X': [Xun],
        'posterior_prob': [p_YgX_ew[Xinds[0],ref_choice]]
    })
    postprobdf = pd.concat((postprobdf,sjdf_ew),ignore_index=True)
    sjdf_iw = pd.DataFrame({
        'strategy': ['ignore-weak'],
        'X': [Xun],
        'posterior_prob': [p_YgX_iw[Xinds[0],ref_choice]]
    })
    postprobdf = pd.concat((postprobdf,sjdf_iw),ignore_index=True)

if save_data_flag:
    fulldf.to_csv(horses_datadir_mid + 'sj-preproc-data.csv',index=False)
    trialset_df.to_csv(horses_datadir_mid + 'trial_set.csv',index=False)
    choiceprobdf.loc[choiceprobdf['condition']=='all'].to_csv(horses_datadir_mid + 'midWS_sj-psychometric-data.csv',index=False)
    choiceprobdf.loc[choiceprobdf['condition']!='all'].to_csv(horses_datadir_mid + 'learning_sj-psychometric-data.csv',index=False)
    postprobdf.to_csv(horses_datadir_mid + 'strat-post-probs.csv',index=False)

C:\Users\parja\AppData\Local\Temp\ipykernel_31780\2512195282.py:113: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  datadf['choice'] = datadf.loc[:,'key_resp_test.keys'] == dist2_side
C:\Users\parja\AppData\Local\Temp\ipykernel_31780\2512195282.py:121: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  datadf['choice'] = datadf.loc[:,'key_resp_test_5.keys'] == dist2_side
C:\Users\parja\AppData\Local\Temp\ipykernel_31780\2512195282.py:113: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice f

### high WS ratio experiment

In [7]:
horses_datadir_high = horses_datadir + "highWS/"
sjlist = pd.read_csv(horses_datadir_high + 'sj_list.csv')

fulldf = pd.DataFrame({
    'subject_ID': [],
    'subject_index': [],
    'exclude_Ixr': [],
    'trial_ID': [],
    'trial_index': [],
    'observation_1': [],
    'observation_2': [],
    'observation_3': [],
    'observation_4': [],
    'observation_5': [],
    'latent_state': [],
    'optimal_choice': [],
    'correct': [],
    'intertrial_duration': [],
    'trial_start_time': [],
    'choice': [],
    'rt': []
})

run1_columns = [
    'trial_index',
    'shape1',
    'shape2',
    'shape3',
    'shape4',
    'shape5',
    'distr',
    'pred_distr',
    'num_completed',
    'key_resp_test.corr',
    'key_resp_6.rt',
    'testing_trial.started',
    'choice', # added in the loop below
    'key_resp_test.rt'
]

renamed_columns = [
    'trial_ID',
    'observation_1',
    'observation_2',
    'observation_3',
    'observation_4',
    'observation_5',
    'latent_state',
    'optimal_choice',
    'trial_index',
    'correct',
    'intertrial_duration',
    'trial_start_time',
    'choice',
    'rt'
]

run1_map = dict(zip(run1_columns,renamed_columns))

shape_map = {
    'shape_1': 1.0,
    'shape_2': 2.0,
    'shape_3': 3.0,
    'shape_4': 4.0
}

shape_map_all = {
    'observation_1': shape_map,
    'observation_2': shape_map,
    'observation_3': shape_map,
    'observation_4': shape_map,
    'observation_5': shape_map
}

dir_list = [
    horses_datadir_high + 'raw/*.csv',
    horses_datadir_high + 'raw/exclude/*.csv',
]

sjindex = 0
dirindex = 0

# get non-excluded subjects
for dir_ in dir_list:
    for fname in glob.glob(dir_):

        sjdf = pd.DataFrame({})

        df = pd.read_csv(fname)
        sjid = df['PROLIFIC_PID'].iloc[0]
        dist2_side = df['dist2_loc'].iloc[0]
        # if df['dist1_loc'].iloc[0] == 'left':
        #     choice_map = dict({'left': 1, 'right': 2})
        # else:
        #     choice_map = dict({'right': 1, 'left': 2})

        # run 1
        datadf = df[df['key_resp_test.keys'].notna()]
        datadf['choice'] = datadf.loc[:,'key_resp_test.keys'] == dist2_side
        datadf = datadf[run1_columns]
        datadf.rename(columns=run1_map,inplace=True)
        sjdf = pd.concat((sjdf,datadf),ignore_index=True)

        sjdf['choice'] = sjdf['choice'].astype(int)
        sjdf['subject_ID'] = sjid
        sjdf['subject_index'] = sjlist[sjlist['subject_ID']==sjid]['subject_index'].iloc[0]
        sjdf['trial_index'] = sjdf['trial_index'] - 100
        if dirindex > 0:
            sjdf['exclude_Ixr'] = 1
        else:
            sjdf['exclude_Ixr'] = 0

        sjindex += 1

        fulldf = pd.concat((fulldf,sjdf),ignore_index=True)

    dirindex += 1

fulldf.replace(shape_map_all,inplace=True)
fulldf['latent_state'] = fulldf['latent_state'] - 1
fulldf['optimal_choice'] = fulldf['optimal_choice'] - 1

fulldf['observation_encoding'] = \
    10**(fulldf['observation_1']-1) + \
    10**(fulldf['observation_2']-1) + \
    10**(fulldf['observation_3']-1) + \
    10**(fulldf['observation_4']-1) + \
    10**(fulldf['observation_5']-1)

fulldf['observation_encoding_ew'] = \
    10**(fulldf['observation_1']>2) + \
    10**(fulldf['observation_2']>2) + \
    10**(fulldf['observation_3']>2) + \
    10**(fulldf['observation_4']>2) + \
    10**(fulldf['observation_5']>2)

no_strong_flag = \
    ~( \
    (fulldf['observation_1']==1) | (fulldf['observation_1']==4) | \
    (fulldf['observation_2']==1) | (fulldf['observation_2']==4) | \
    (fulldf['observation_3']==1) | (fulldf['observation_3']==4) | \
    (fulldf['observation_4']==1) | (fulldf['observation_4']==4) | \
    (fulldf['observation_5']==1) | (fulldf['observation_5']==4) \
        )

fulldf['observation_encoding_iw'] = \
    (10**(fulldf['observation_1']-1)) * ((fulldf['observation_1']==1) | (fulldf['observation_1']==4)+no_strong_flag) + \
    (10**(fulldf['observation_2']-1)) * ((fulldf['observation_2']==1) | (fulldf['observation_2']==4)+no_strong_flag) + \
    (10**(fulldf['observation_3']-1)) * ((fulldf['observation_3']==1) | (fulldf['observation_3']==4)+no_strong_flag) + \
    (10**(fulldf['observation_4']-1)) * ((fulldf['observation_4']==1) | (fulldf['observation_4']==4)+no_strong_flag) + \
    (10**(fulldf['observation_5']-1)) * ((fulldf['observation_5']==1) | (fulldf['observation_5']==4)+no_strong_flag)

fulldf.sort_values(by=['subject_index','trial_index'],inplace=True,ignore_index=True)

# make a csv of the trial set (horses and shape combinations) that every subject saw
trialset_df = fulldf.loc[(fulldf['subject_index']==30)][['trial_ID','observation_encoding','latent_state']].sort_values(by='trial_ID',ignore_index=True).copy()

# make a csv that gives the choice probabilities for each unique X (shape combination) for each subject
choiceprobdf = pd.DataFrame({
    'subject_ID': [],
    'subject_index': [],
    'X': [],
    'Ntrials': [],  # number of trials of a given unique X
    'posterior_prob': [],
    'choice_prob': []
})

Xemp = utils.split_to_four_digits(trialset_df['observation_encoding'].to_numpy())
Xemp_enc = trialset_df['observation_encoding'].to_numpy()
Yemp = trialset_df['latent_state'].to_numpy()
p_YgX = utils.P_horse_g_shapecomb(Xemp,paramdict['highWS']['weakLLR'],paramdict['highWS']['WSratio'],paramdict['highWS']['p1'],p_Y)
Xunique = np.unique(Xemp_enc)

for sjind in fulldf['subject_index'].unique():

    sjconds = fulldf['subject_index']==sjind
    sjid = fulldf[sjconds]['subject_ID'].iloc[0]

    Xemp_sj = utils.split_to_four_digits(fulldf[sjconds]['observation_encoding'].to_numpy())
    Xemp_sj_enc = fulldf[sjconds]['observation_encoding'].to_numpy()
    Yemp_sj = fulldf[sjconds]['latent_state'].to_numpy()
    choices = fulldf[sjconds]['choice'].to_numpy()

    # for Xun in Xunique:
    for Xun in np.unique(Xemp_sj_enc):

        Xinds = np.where(Xemp_sj_enc == Xun)[0]
        Xinds2 = np.where(Xemp_enc == Xun)[0]

        choice_prob = np.mean(choices[Xinds]==ref_choice)

        sjdf = pd.DataFrame({
            'subject_ID': [sjid],
            'subject_index': [sjind],
            'X': [Xun],
            'Ntrials': [len(Xinds)],
            'posterior_prob': [p_YgX[Xinds2[0],ref_choice]],
            'choice_prob': [choice_prob]
        })

        choiceprobdf = pd.concat((choiceprobdf,sjdf),ignore_index=True)


# make csv giving the posterior probability of the hidden state (horse) given each unique X for a given strategy - e.g., optimal strategy, or equal-weights strategy, etc.
postprobdf = pd.DataFrame({
    'strategy': [],
    'X': [],
    'posterior_prob': []
})

p_YgX_ew = utils.P_horse_g_shapecomb(Xemp,paramdict['highWS']['weakLLR'],paramdict['highWS']['WSratio'],paramdict['highWS']['p1'],p_Y,utils.P_shapecomb_g_horse_ew)
p_YgX_iw = utils.P_horse_g_shapecomb(Xemp,paramdict['highWS']['weakLLR'],paramdict['highWS']['WSratio'],paramdict['highWS']['p1'],p_Y,utils.P_shapecomb_g_horse_iw)

for Xun in Xunique:
    Xinds = np.where(Xemp_enc == Xun)[0]
    sjdf = pd.DataFrame({
        'strategy': ['fully-optimal'],
        'X': [Xun],
        'posterior_prob': [p_YgX[Xinds[0],ref_choice]]
    })
    postprobdf = pd.concat((postprobdf,sjdf),ignore_index=True)
    sjdf_ew = pd.DataFrame({
        'strategy': ['equal-weights'],
        'X': [Xun],
        'posterior_prob': [p_YgX_ew[Xinds[0],ref_choice]]
    })
    postprobdf = pd.concat((postprobdf,sjdf_ew),ignore_index=True)
    sjdf_iw = pd.DataFrame({
        'strategy': ['ignore-weak'],
        'X': [Xun],
        'posterior_prob': [p_YgX_iw[Xinds[0],ref_choice]]
    })
    postprobdf = pd.concat((postprobdf,sjdf_iw),ignore_index=True)

if save_data_flag:
    fulldf.to_csv(horses_datadir_high + 'sj-preproc-data.csv',index=False)
    trialset_df.to_csv(horses_datadir_high + 'trial_set.csv',index=False)
    choiceprobdf.to_csv(horses_datadir_high + 'sj-psychometric-data.csv',index=False)
    postprobdf.to_csv(horses_datadir_high + 'strat-post-probs.csv',index=False)

C:\Users\parja\AppData\Local\Temp\ipykernel_31780\1114845516.py:99: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  datadf['choice'] = datadf.loc[:,'key_resp_test.keys'] == dist2_side
C:\Users\parja\AppData\Local\Temp\ipykernel_31780\1114845516.py:99: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  datadf['choice'] = datadf.loc[:,'key_resp_test.keys'] == dist2_side
C:\Users\parja\AppData\Local\Temp\ipykernel_31780\1114845516.py:99: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a

### Speed-accuracy experiment

In [8]:
horses_datadir_sa = horses_datadir + "speed_accuracy/"
sjlist = pd.read_csv(horses_datadir_sa + 'sj_list.csv')

fulldf = pd.DataFrame({
    'subject_ID': [],
    'subject_index': [],
    'exclude_Ixr': [],
    'condition': [],
    'trial_ID': [],
    'trial_index': [],
    'observation_1': [],
    'observation_2': [],
    'observation_3': [],
    'observation_4': [],
    'observation_5': [],
    'latent_state': [],
    'optimal_choice': [],
    'correct': [],
    'intertrial_duration': [],
    'trial_start_time': [],
    'choice': [],
    'rt': []
})

run1_columns = [
    'time',
    'trial_index',
    'shape1',
    'shape2',
    'shape3',
    'shape4',
    'shape5',
    'distr',
    'pred_distr',
    'num_completed',
    'key_resp_test.corr',
    'key_resp_6.rt',
    'testing_trial.started',
    'choice', # added in the loop below
    'key_resp_test.rt'
]

renamed_columns = [
    'condition',
    'trial_ID',
    'observation_1',
    'observation_2',
    'observation_3',
    'observation_4',
    'observation_5',
    'latent_state',
    'optimal_choice',
    'trial_index',
    'correct',
    'intertrial_duration',
    'trial_start_time',
    'choice',
    'rt'
]

run1_map = dict(zip(run1_columns,renamed_columns))

shape_map = {
    'shape_1': 1.0,
    'shape_2': 2.0,
    'shape_3': 3.0,
    'shape_4': 4.0
}

shape_map_all = {
    'observation_1': shape_map,
    'observation_2': shape_map,
    'observation_3': shape_map,
    'observation_4': shape_map,
    'observation_5': shape_map
}

condition_map = {
    'condition': {
        0.75: 'fast',
        1.75: 'slow'
    }
}

sjindex = 0

# get non-excluded subjects
for fname in glob.glob(horses_datadir_sa + 'raw/*.csv'):

    sjdf = pd.DataFrame({})

    df = pd.read_csv(fname)
    sjid = df['PROLIFIC_PID'].iloc[0]
    dist2_side = df['dist2_loc'].iloc[0]
    # if df['dist1_loc'].iloc[0] == 'left':
    #     choice_map = dict({'left': 1, 'right': 2})
    # else:
    #     choice_map = dict({'right': 1, 'left': 2})

    # run 1
    datadf = df[df['key_resp_test.keys'].notna()]
    datadf['choice'] = datadf.loc[:,'key_resp_test.keys'] == dist2_side
    datadf = datadf[run1_columns]
    datadf.rename(columns=run1_map,inplace=True)
    sjdf = pd.concat((sjdf,datadf),ignore_index=True)

    sjdf['choice'] = sjdf['choice'].astype(int)
    sjdf['subject_ID'] = sjid
    sjdf['subject_index'] = sjlist[sjlist['subject_ID']==sjid]['subject_index'].iloc[0]
    sjdf['exclude_Ixr'] = 0
    sjdf['trial_index'] = sjdf['trial_index'] - 100

    sjindex += 1

    fulldf = pd.concat((fulldf,sjdf),ignore_index=True)

# get subjects excluded for inference capacity < 0.05 for both conditions
for fname in glob.glob(horses_datadir_sa + 'raw/exclude/*.csv'):

    sjdf = pd.DataFrame({})

    df = pd.read_csv(fname)
    sjid = df['PROLIFIC_PID'].iloc[0]
    dist2_side = df['dist2_loc'].iloc[0]
    # if df['dist1_loc'].iloc[0] == 'left':
    #     choice_map = dict({'left': 1, 'right': 2})
    # else:
    #     choice_map = dict({'right': 1, 'left': 2})

    # run 1
    datadf = df[df['key_resp_test.keys'].notna()]
    datadf['choice'] = datadf.loc[:,'key_resp_test.keys'] == dist2_side
    datadf = datadf[run1_columns]
    datadf.rename(columns=run1_map,inplace=True)
    sjdf = pd.concat((sjdf,datadf),ignore_index=True)

    sjdf['choice'] = sjdf['choice'].astype(int)
    sjdf['subject_ID'] = sjid
    sjdf['subject_index'] = sjlist[sjlist['subject_ID']==sjid]['subject_index'].iloc[0]
    sjdf['exclude_Ixr'] = 1
    sjdf['trial_index'] = sjdf['trial_index'] - 100

    sjindex += 1

    fulldf = pd.concat((fulldf,sjdf),ignore_index=True)

fulldf.replace(shape_map_all,inplace=True)
fulldf.replace(condition_map,inplace=True)
fulldf['latent_state'] = fulldf['latent_state'] - 1
fulldf['optimal_choice'] = fulldf['optimal_choice'] - 1

fulldf['observation_encoding'] = \
    10**(fulldf['observation_1']-1) + \
    10**(fulldf['observation_2']-1) + \
    10**(fulldf['observation_3']-1) + \
    10**(fulldf['observation_4']-1) + \
    10**(fulldf['observation_5']-1)

fulldf['observation_encoding_ew'] = \
    10**(fulldf['observation_1']>2) + \
    10**(fulldf['observation_2']>2) + \
    10**(fulldf['observation_3']>2) + \
    10**(fulldf['observation_4']>2) + \
    10**(fulldf['observation_5']>2)

no_strong_flag = \
    ~( \
    (fulldf['observation_1']==1) | (fulldf['observation_1']==4) | \
    (fulldf['observation_2']==1) | (fulldf['observation_2']==4) | \
    (fulldf['observation_3']==1) | (fulldf['observation_3']==4) | \
    (fulldf['observation_4']==1) | (fulldf['observation_4']==4) | \
    (fulldf['observation_5']==1) | (fulldf['observation_5']==4) \
        )

fulldf['observation_encoding_iw'] = \
    (10**(fulldf['observation_1']-1)) * ((fulldf['observation_1']==1) | (fulldf['observation_1']==4)+no_strong_flag) + \
    (10**(fulldf['observation_2']-1)) * ((fulldf['observation_2']==1) | (fulldf['observation_2']==4)+no_strong_flag) + \
    (10**(fulldf['observation_3']-1)) * ((fulldf['observation_3']==1) | (fulldf['observation_3']==4)+no_strong_flag) + \
    (10**(fulldf['observation_4']-1)) * ((fulldf['observation_4']==1) | (fulldf['observation_4']==4)+no_strong_flag) + \
    (10**(fulldf['observation_5']-1)) * ((fulldf['observation_5']==1) | (fulldf['observation_5']==4)+no_strong_flag)

fulldf.sort_values(by=['subject_index','trial_index'],inplace=True,ignore_index=True)

# make a csv of the trial set (horses and shape combinations) that every subject saw
trialset_df = fulldf.loc[(fulldf['subject_index']==0) & (fulldf['condition']=='slow')][['trial_ID','observation_encoding','latent_state']].sort_values(by='trial_ID',ignore_index=True).copy()

# make a csv that gives the choice probabilities for each unique X (shape combination) for each subject
choiceprobdf = pd.DataFrame({
    'subject_ID': [],
    'subject_index': [],
    'condition': [],
    'X': [],
    'Ntrials': [],  # number of trials of a given unique X
    'posterior_prob': [],
    'choice_prob': []
})

Xemp = utils.split_to_four_digits(trialset_df['observation_encoding'].to_numpy())
Xemp_enc = trialset_df['observation_encoding'].to_numpy()
Yemp = trialset_df['latent_state'].to_numpy()
p_YgX = utils.P_horse_g_shapecomb(Xemp,paramdict['midWS']['weakLLR'],paramdict['midWS']['WSratio'],paramdict['midWS']['p1'],p_Y)
Xunique = np.unique(Xemp_enc)

for sjind in fulldf['subject_index'].unique():

    for cond in ['fast', 'slow']:

        sjconds = (fulldf['subject_index']==sjind) & (fulldf['condition']==cond)
        sjid = fulldf[sjconds]['subject_ID'].iloc[0]

        Xemp_sj = utils.split_to_four_digits(fulldf[sjconds]['observation_encoding'].to_numpy())
        Xemp_sj_enc = fulldf[sjconds]['observation_encoding'].to_numpy()
        choices = fulldf[sjconds]['choice'].to_numpy()

        # for Xun in Xunique:
        for Xun in np.unique(Xemp_sj_enc):

            Xinds = np.where(Xemp_sj_enc == Xun)[0]
            Xinds2 = np.where(Xemp_enc == Xun)[0]

            choice_prob = np.mean(choices[Xinds]==ref_choice)

            sjdf = pd.DataFrame({
                'subject_ID': [sjid],
                'subject_index': [sjind],
                'condition': [cond],
                'X': [Xun],
                'Ntrials': [len(Xinds)],
                'posterior_prob': [p_YgX[Xinds2[0],ref_choice]],
                'choice_prob': [choice_prob]
            })

            choiceprobdf = pd.concat((choiceprobdf,sjdf),ignore_index=True)

# make csv giving the posterior probability of the hidden state (horse) given each unique X for a given strategy - e.g., optimal strategy, or equal-weights strategy, etc.
postprobdf = pd.DataFrame({
    'strategy': [],
    'X': [],
    'posterior_prob': []
})

p_YgX_ew = utils.P_horse_g_shapecomb(Xemp,paramdict['midWS']['weakLLR'],paramdict['midWS']['WSratio'],paramdict['midWS']['p1'],p_Y,utils.P_shapecomb_g_horse_ew)
p_YgX_iw = utils.P_horse_g_shapecomb(Xemp,paramdict['midWS']['weakLLR'],paramdict['midWS']['WSratio'],paramdict['midWS']['p1'],p_Y,utils.P_shapecomb_g_horse_iw)

for Xun in Xunique:
    Xinds = np.where(Xemp_enc == Xun)[0]
    sjdf = pd.DataFrame({
        'strategy': ['fully-optimal'],
        'X': [Xun],
        'posterior_prob': [p_YgX[Xinds[0],ref_choice]]
    })
    postprobdf = pd.concat((postprobdf,sjdf),ignore_index=True)
    sjdf_ew = pd.DataFrame({
        'strategy': ['equal-weights'],
        'X': [Xun],
        'posterior_prob': [p_YgX_ew[Xinds[0],ref_choice]]
    })
    postprobdf = pd.concat((postprobdf,sjdf_ew),ignore_index=True)
    sjdf_iw = pd.DataFrame({
        'strategy': ['ignore-weak'],
        'X': [Xun],
        'posterior_prob': [p_YgX_iw[Xinds[0],ref_choice]]
    })
    postprobdf = pd.concat((postprobdf,sjdf_iw),ignore_index=True)

if save_data_flag:
    fulldf.to_csv(horses_datadir_sa + 'sj-preproc-data.csv',index=False)
    choiceprobdf.to_csv(horses_datadir_sa + 'sj-psychometric-data.csv',index=False)
    postprobdf.to_csv(horses_datadir_sa + 'strat-post-probs.csv',index=False)

C:\Users\parja\AppData\Local\Temp\ipykernel_31780\3293277853.py:102: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  datadf['choice'] = datadf.loc[:,'key_resp_test.keys'] == dist2_side
C:\Users\parja\AppData\Local\Temp\ipykernel_31780\3293277853.py:102: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  datadf['choice'] = datadf.loc[:,'key_resp_test.keys'] == dist2_side
C:\Users\parja\AppData\Local\Temp\ipykernel_31780\3293277853.py:102: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice fro

### Reward experiment

In [9]:
horses_datadir_rew = horses_datadir + "reward/"
sjlist = pd.read_csv(horses_datadir_rew + 'sj_list.csv')

fulldf = pd.DataFrame({
    'subject_ID': [],
    'subject_index': [],
    'exclude_Ixr': [],
    'condition': [],
    'block': [],
    'trial_ID': [],
    'trial_index': [],
    'observation_1': [],
    'observation_2': [],
    'observation_3': [],
    'observation_4': [],
    'observation_5': [],
    'latent_state': [],
    'optimal_choice': [],
    'correct': [],
    'intertrial_duration': [],
    'trial_start_time': [],
    'choice': [],
    'rt': []
})

run1_columns = [
    'trial_index',
    'shape1',
    'shape2',
    'shape3',
    'shape4',
    'shape5',
    'distr',
    'pred_distr',
    'num_completed_1',
    'key_resp_test.corr',
    'key_proceed_1.rt',
    'testing_trial.started',
    'choice', # added in the loop below
    'key_resp_test.rt'
]

run2_columns = [
    'trial_index',
    'shape1',
    'shape2',
    'shape3',
    'shape4',
    'shape5',
    'distr',
    'pred_distr',
    'num_completed_2',
    'key_resp_test_5.corr',
    'key_proceed_2.rt',
    'testing_trial_2.started',
    'choice',
    'key_resp_test_5.rt'
]

renamed_columns = [
    'trial_ID',
    'observation_1',
    'observation_2',
    'observation_3',
    'observation_4',
    'observation_5',
    'latent_state',
    'optimal_choice',
    'trial_index',
    'correct',
    'intertrial_duration',
    'trial_start_time',
    'choice',
    'rt'
]

run1_map = dict(zip(run1_columns,renamed_columns))
run2_map = dict(zip(run2_columns,renamed_columns))

shape_map = {
    'shape_1': 1.0,
    'shape_2': 2.0,
    'shape_3': 3.0,
    'shape_4': 4.0
}

shape_map_all = {
    'observation_1': shape_map,
    'observation_2': shape_map,
    'observation_3': shape_map,
    'observation_4': shape_map,
    'observation_5': shape_map
}

dir_list = [
    horses_datadir_rew + 'raw/low_first/*.csv',
    horses_datadir_rew + 'raw/high_first/*.csv',
    horses_datadir_rew + 'raw/low_first/exclude/*.csv',
    horses_datadir_rew + 'raw/high_first/exclude/*.csv'
]

sjindex = 0
dirindex = 0

# get non-excluded subjects
for dir_ in dir_list:
    for fname in glob.glob(dir_):

        sjdf = pd.DataFrame({})

        df = pd.read_csv(fname)
        sjid = df['PROLIFIC_PID'].iloc[0]
        dist2_side = df['dist2_loc'].iloc[0]
        # if df['dist1_loc'].iloc[0] == 'left':
        #     choice_map = dict({'left': 1, 'right': 2})
        # else:
        #     choice_map = dict({'right': 1, 'left': 2})

        # run 1
        datadf = df[df['key_resp_test.keys'].notna()]
        datadf['choice'] = datadf.loc[:,'key_resp_test.keys'] == dist2_side
        datadf = datadf[run1_columns]
        datadf.rename(columns=run1_map,inplace=True)
        datadf['block'] = 1
        if dirindex % 2 == 0:
            datadf['condition'] = 'low_reward'
        else:
            datadf['condition'] = 'high_reward'
        sjdf = pd.concat((sjdf,datadf),ignore_index=True)

        # run 2
        datadf = df[df['key_resp_test_5.keys'].notna()]
        datadf['choice'] = datadf.loc[:,'key_resp_test_5.keys'] == dist2_side
        datadf = datadf[run2_columns]
        datadf.rename(columns=run2_map,inplace=True)
        datadf['block'] = 2
        if dirindex % 2 == 0:
            datadf['condition'] = 'high_reward'
        else:
            datadf['condition'] = 'low_reward'
        sjdf = pd.concat((sjdf,datadf),ignore_index=True)

        sjdf['choice'] = sjdf['choice'].astype(int)
        sjdf['subject_ID'] = sjid
        sjdf['subject_index'] = sjlist[sjlist['subject_ID']==sjid]['subject_index'].iloc[0]
        if dirindex > 1:
            sjdf['exclude_Ixr'] = 1
        else:
            sjdf['exclude_Ixr'] = 0

        sjindex += 1

        fulldf = pd.concat((fulldf,sjdf),ignore_index=True)

    dirindex += 1


fulldf.replace(shape_map_all,inplace=True)
fulldf['latent_state'] = fulldf['latent_state'] - 1
fulldf['optimal_choice'] = fulldf['optimal_choice'] - 1

fulldf['observation_encoding'] = \
    10**(fulldf['observation_1']-1) + \
    10**(fulldf['observation_2']-1) + \
    10**(fulldf['observation_3']-1) + \
    10**(fulldf['observation_4']-1) + \
    10**(fulldf['observation_5']-1)

fulldf['observation_encoding_ew'] = \
    10**(fulldf['observation_1']>2) + \
    10**(fulldf['observation_2']>2) + \
    10**(fulldf['observation_3']>2) + \
    10**(fulldf['observation_4']>2) + \
    10**(fulldf['observation_5']>2)

no_strong_flag = \
    ~( \
    (fulldf['observation_1']==1) | (fulldf['observation_1']==4) | \
    (fulldf['observation_2']==1) | (fulldf['observation_2']==4) | \
    (fulldf['observation_3']==1) | (fulldf['observation_3']==4) | \
    (fulldf['observation_4']==1) | (fulldf['observation_4']==4) | \
    (fulldf['observation_5']==1) | (fulldf['observation_5']==4) \
        )

fulldf['observation_encoding_iw'] = \
    (10**(fulldf['observation_1']-1)) * ((fulldf['observation_1']==1) | (fulldf['observation_1']==4)+no_strong_flag) + \
    (10**(fulldf['observation_2']-1)) * ((fulldf['observation_2']==1) | (fulldf['observation_2']==4)+no_strong_flag) + \
    (10**(fulldf['observation_3']-1)) * ((fulldf['observation_3']==1) | (fulldf['observation_3']==4)+no_strong_flag) + \
    (10**(fulldf['observation_4']-1)) * ((fulldf['observation_4']==1) | (fulldf['observation_4']==4)+no_strong_flag) + \
    (10**(fulldf['observation_5']-1)) * ((fulldf['observation_5']==1) | (fulldf['observation_5']==4)+no_strong_flag)

fulldf.sort_values(by=['subject_index','block','trial_index'],inplace=True,ignore_index=True)

# make a csv of the trial set (horses and shape combinations) that every subject saw
trialset_df = fulldf.loc[(fulldf['subject_index']==0) & (fulldf['condition']=='low_reward')][['trial_ID','observation_encoding','latent_state']].sort_values(by='trial_ID',ignore_index=True).copy()

# make a csv that gives the choice probabilities for each unique X (shape combination) for each subject
choiceprobdf = pd.DataFrame({
    'subject_ID': [],
    'subject_index': [],
    'condition': [],
    'block': [],
    'X': [],
    'Ntrials': [],  # number of trials of a given unique X
    'posterior_prob': [],
    'choice_prob': []
})

Xemp = utils.split_to_four_digits(trialset_df['observation_encoding'].to_numpy())
Xemp_enc = trialset_df['observation_encoding'].to_numpy()
Yemp = trialset_df['latent_state'].to_numpy()
p_YgX = utils.P_horse_g_shapecomb(Xemp,paramdict['midWS']['weakLLR'],paramdict['midWS']['WSratio'],paramdict['midWS']['p1'],p_Y)
Xunique = np.unique(Xemp_enc)

for sjind in fulldf['subject_index'].unique():

    for cond in ['low_reward', 'high_reward']:

        sjconds = (fulldf['subject_index']==sjind) & (fulldf['condition']==cond)
        sjid = fulldf[sjconds]['subject_ID'].iloc[0]

        Xemp_sj = utils.split_to_four_digits(fulldf[sjconds]['observation_encoding'].to_numpy())
        Xemp_sj_enc = fulldf[sjconds]['observation_encoding'].to_numpy()
        choices = fulldf[sjconds]['choice'].to_numpy()

        # for Xun in Xunique:
        for Xun in np.unique(Xemp_sj_enc):

            Xinds = np.where(Xemp_sj_enc == Xun)[0]
            Xinds2 = np.where(Xemp_enc == Xun)[0]

            choice_prob = np.mean(choices[Xinds]==ref_choice)

            sjdf = pd.DataFrame({
                'subject_ID': [sjid],
                'subject_index': [sjind],
                'condition': [cond],
                'block': [fulldf[sjconds]['block'].iloc[0]],
                'X': [Xun],
                'Ntrials': [len(Xinds)],
                'posterior_prob': [p_YgX[Xinds2[0],ref_choice]],
                'choice_prob': [choice_prob]
            })

            choiceprobdf = pd.concat((choiceprobdf,sjdf),ignore_index=True)

# make csv giving the posterior probability of the hidden state (horse) given each unique X for a given strategy - e.g., optimal strategy, or equal-weights strategy, etc.
postprobdf = pd.DataFrame({
    'strategy': [],
    'X': [],
    'posterior_prob': []
})

p_YgX_ew = utils.P_horse_g_shapecomb(Xemp,paramdict['midWS']['weakLLR'],paramdict['midWS']['WSratio'],paramdict['midWS']['p1'],p_Y,utils.P_shapecomb_g_horse_ew)
p_YgX_iw = utils.P_horse_g_shapecomb(Xemp,paramdict['midWS']['weakLLR'],paramdict['midWS']['WSratio'],paramdict['midWS']['p1'],p_Y,utils.P_shapecomb_g_horse_iw)

for Xun in Xunique:
    Xinds = np.where(Xemp_enc == Xun)[0]
    sjdf = pd.DataFrame({
        'strategy': ['fully-optimal'],
        'X': [Xun],
        'posterior_prob': [p_YgX[Xinds[0],ref_choice]]
    })
    postprobdf = pd.concat((postprobdf,sjdf),ignore_index=True)
    sjdf_ew = pd.DataFrame({
        'strategy': ['equal-weights'],
        'X': [Xun],
        'posterior_prob': [p_YgX_ew[Xinds[0],ref_choice]]
    })
    postprobdf = pd.concat((postprobdf,sjdf_ew),ignore_index=True)
    sjdf_iw = pd.DataFrame({
        'strategy': ['ignore-weak'],
        'X': [Xun],
        'posterior_prob': [p_YgX_iw[Xinds[0],ref_choice]]
    })
    postprobdf = pd.concat((postprobdf,sjdf_iw),ignore_index=True)

if save_data_flag:
    fulldf.to_csv(horses_datadir_rew + 'sj-preproc-data.csv',index=False)
    choiceprobdf.to_csv(horses_datadir_rew + 'sj-psychometric-data.csv',index=False)
    postprobdf.to_csv(horses_datadir_rew + 'strat-post-probs.csv',index=False)

C:\Users\parja\AppData\Local\Temp\ipykernel_31780\873788496.py:121: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  datadf['choice'] = datadf.loc[:,'key_resp_test.keys'] == dist2_side
C:\Users\parja\AppData\Local\Temp\ipykernel_31780\873788496.py:133: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  datadf['choice'] = datadf.loc[:,'key_resp_test_5.keys'] == dist2_side
C:\Users\parja\AppData\Local\Temp\ipykernel_31780\873788496.py:121: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from